In [22]:
from pathlib import Path

data_path = Path("data/")

image_path = data_path / "desert101"

In [23]:
image_path

WindowsPath('data/desert101')

In [24]:
import os 


In [25]:
def check_data(dir_path):
    for dirpath, dirnames, filenames in os.walk(dir_path):
        print(f"# of directories: {len(dirnames)} and {len(filenames)} images in {dirpath}")

In [26]:
check_data(image_path)

# of directories: 2 and 0 images in data\desert101
# of directories: 4 and 1 images in data\desert101\test
# of directories: 0 and 20 images in data\desert101\test\baklava
# of directories: 0 and 20 images in data\desert101\test\cannoli
# of directories: 0 and 20 images in data\desert101\test\cup_cakes
# of directories: 0 and 20 images in data\desert101\test\donuts
# of directories: 4 and 1 images in data\desert101\train
# of directories: 0 and 80 images in data\desert101\train\baklava
# of directories: 0 and 80 images in data\desert101\train\cannoli
# of directories: 0 and 80 images in data\desert101\train\cup_cakes
# of directories: 0 and 80 images in data\desert101\train\donuts


In [27]:
train_dir = image_path / "train"
test_dir = image_path / "test"

In [28]:
train_dir

WindowsPath('data/desert101/train')

In [29]:
test_dir

WindowsPath('data/desert101/test')

In [30]:
from PIL import Image
import random


In [ ]:
random.seed(42)

image_path_list = list(image_path.glob("*/*/*.jpg"))
random_image = random.choice(image_path_list)
img = Image.open(random_image)
img.show()


In [39]:
random_image

WindowsPath('data/desert101/train/donuts/2290677.jpg')

In [47]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [50]:
data_transforms = transforms.Compose(
    [
        transforms.Resize(size = (64, 64)),
        transforms.RandomHorizontalFlip(p = 0.4),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
    ])

In [51]:
train_data = datasets.ImageFolder(root = train_dir, transform = data_transforms)
test_data = datasets.ImageFolder(root = test_dir, transform = data_transforms)

In [52]:
train_data

Dataset ImageFolder
    Number of datapoints: 316
    Root location: data\desert101\train
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               RandomHorizontalFlip(p=0.4)
               TrivialAugmentWide(num_magnitude_bins=31, interpolation=InterpolationMode.NEAREST, fill=None)
               ToTensor()
           )

In [53]:
test_data

Dataset ImageFolder
    Number of datapoints: 77
    Root location: data\desert101\test
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               RandomHorizontalFlip(p=0.4)
               TrivialAugmentWide(num_magnitude_bins=31, interpolation=InterpolationMode.NEAREST, fill=None)
               ToTensor()
           )

In [ ]:
class_names = train_data.classes

['baklava', 'cannoli', 'cup_cakes', 'donuts']

In [55]:
len(train_data), len(test_data)

(316, 77)

In [58]:
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

In [59]:
NUM_WORKERS

20

In [61]:
train_dataloader = DataLoader(dataset=train_data,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              num_workers=NUM_WORKERS)

test_dataloader = DataLoader(dataset=test_data,
                             batch_size=BATCH_SIZE,
                             shuffle=False,
                             num_workers=NUM_WORKERS)

In [63]:
train_dataloader.dataset[0][0]

tensor([[[0.0196, 0.0157, 0.0275,  ..., 0.0510, 0.0863, 0.0941],
         [0.0235, 0.0157, 0.0510,  ..., 0.0392, 0.0627, 0.1333],
         [0.0588, 0.0235, 0.0824,  ..., 0.0667, 0.0549, 0.1922],
         ...,
         [0.4353, 0.5098, 0.5451,  ..., 0.9412, 0.9176, 0.8745],
         [0.4980, 0.4471, 0.4157,  ..., 0.9647, 0.9529, 0.8627],
         [0.4902, 0.5961, 0.5451,  ..., 0.8824, 0.9020, 0.8549]],

        [[0.0353, 0.0353, 0.0471,  ..., 0.0745, 0.1176, 0.1216],
         [0.0471, 0.0392, 0.0706,  ..., 0.0667, 0.0863, 0.1961],
         [0.0706, 0.0431, 0.0941,  ..., 0.0863, 0.0824, 0.2824],
         ...,
         [0.2118, 0.2824, 0.3451,  ..., 0.9137, 0.8314, 0.6706],
         [0.2941, 0.2706, 0.2431,  ..., 0.9216, 0.9098, 0.7098],
         [0.3529, 0.4784, 0.4392,  ..., 0.8196, 0.8196, 0.6706]],

        [[0.0549, 0.0510, 0.0667,  ..., 0.2431, 0.4314, 0.4157],
         [0.0941, 0.0745, 0.1804,  ..., 0.1804, 0.2824, 0.4980],
         [0.0784, 0.0784, 0.2706,  ..., 0.1922, 0.2549, 0.

In [ ]:
from torch import nn
class DesertClassifier(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride = 2)
        )
        
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride = 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units * 16 * 16, out_features=output_shape)
        )

    def forward(self, x):
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))